# MegatronBridge API — Colab Quickstart

This notebook runs the full MegatronBridge API stack on a Colab GPU and walks through:
1. Environment verification
2. Installing dependencies (with known Colab compatibility fixes)
3. Starting the API server
4. Submitting a checkpoint import job
5. Streaming real-time logs via WebSocket
6. Monitoring GPU telemetry via the progress stream

**Verified environment (April 2026)**
```
Python:  3.12.13
PyTorch: 2.10.0+cu128
CUDA:    12.8 toolkit  (driver reports 13.0 — use core_cu12 TE variant)
GPU:     NVIDIA RTX PRO 6000 Blackwell Server Edition (95.6 GB)
```

**Required runtime: H100 or A100**  
Runtime → Change runtime type → Hardware accelerator: H100

[![GitHub](https://img.shields.io/badge/GitHub-Nvidia--megatron--bridge--API-black)](https://github.com/Doondi-Ashlesh/Nvidia-megatron-bridge-API)

## Step 1 — Verify environment

Requirements: Python ≥ 3.12, PyTorch ≥ 2.7, CUDA ≥ 12.8

In [ ]:
import sys
import subprocess
import torch

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print()

py_ok   = sys.version_info >= (3, 12)
pt_ok   = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) >= (2, 7)
cuda_ok = torch.version.cuda is not None and float(torch.version.cuda) >= 12.8

print(f"Python ≥ 3.12:  {'✅' if py_ok   else '❌'}")
print(f"PyTorch ≥ 2.7:  {'✅' if pt_ok   else '❌'}")
print(f"CUDA ≥ 12.8:    {'✅' if cuda_ok else '❌'}")

if not all([py_ok, pt_ok, cuda_ok]):
    raise RuntimeError("Requirements not met. Switch to H100/A100 runtime: Runtime → Change runtime type")

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print()
print(result.stdout)

## Step 2 — Clone repository

In [ ]:
import os

if not os.path.exists("/content/Nvidia-megatron-bridge-API"):
    !git clone https://github.com/Doondi-Ashlesh/Nvidia-megatron-bridge-API /content/Nvidia-megatron-bridge-API

os.chdir("/content/Nvidia-megatron-bridge-API")
print("Working directory:", os.getcwd())

## Step 3 — Install dependencies

Several Colab-specific version pins are required. Each cell below has a comment explaining why.

In [ ]:
# API server dependencies — fast, CPU-only, no conflicts
!pip install -e ".[dev]" -q
print("✅ API server dependencies installed")

In [ ]:
# megatron-bridge: install with --no-deps to avoid megatron-core[dev,mlm]
# pulling in dev dependencies that conflict with Colab's environment.
# We install megatron-core separately without the dev extras.
!pip install setuptools pybind11 wheel_stub -q
!pip install megatron-bridge --no-deps -q
!pip install megatron-core -q
print("✅ megatron-bridge installed")

In [ ]:
# transformer-engine: use core_cu12 variant.
# Although the driver reports CUDA 13.0, the installed CUDA toolkit is 12.8
# and only libcublas.so.12 is present. core_cu13 fails with
# 'libcublas.so.13: cannot open shared object file'.
# Also pin <2.13.0 — megatron-bridge 0.3.1 requires <2.13.0.
!pip install "transformer-engine[core_cu12,pytorch]>=2.10.0,<2.13.0" -q
print("✅ transformer-engine installed")

In [ ]:
# transformers: megatron-bridge 0.3.1 requires <5.0.0.
# Colab ships 5.0.0 exactly, which is incompatible.
!pip install "transformers<5.0.0" -q
print("✅ transformers pinned to <5.0.0")

## Step 4 — Configure environment

In [ ]:
import os

os.environ["CHECKPOINTS_ROOT"] = "/content/checkpoints"
os.environ["LOGS_ROOT"]        = "/content/logs"
os.environ["DATABASE_URL"]     = "sqlite+aiosqlite:////content/megatronbridge.db"
os.environ["LOG_LEVEL"]        = "INFO"

# CUDA library path — required so the worker subprocess can find libcublas.so.12.
# Without this the job fails with 'libcublas.so.12: cannot open shared object file'.
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/cuda-12.8/targets/x86_64-linux/lib:"
    "/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

!mkdir -p /content/checkpoints /content/logs

print("✅ Environment configured")
print(f"  Checkpoints:      {os.environ['CHECKPOINTS_ROOT']}")
print(f"  Logs:             {os.environ['LOGS_ROOT']}")
print(f"  Database:         {os.environ['DATABASE_URL']}")
print(f"  LD_LIBRARY_PATH:  {os.environ['LD_LIBRARY_PATH'][:80]}...")

## Step 5 — Start the API server

In [ ]:
import subprocess
import time
import urllib.request

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),   # passes LD_LIBRARY_PATH to all child processes
)

print("Starting API server", end="")
for _ in range(15):
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        resp = urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print(f"\n✅ Server is up: {resp.read().decode()}")
        break
    except Exception:
        continue
else:
    out = server_proc.stdout.read1(4096)
    print(f"\n❌ Server did not start:\n{out.decode()}")

In [ ]:
import requests
import json

BASE = "http://localhost:8000"

resp = requests.get(f"{BASE}/v1/system/info")
print(json.dumps(resp.json(), indent=2))
# peak_tflops should be populated for H100, A100, RTX PRO 6000 Blackwell

## Step 6 — Expose publicly via ngrok (optional)

Skip this step if you only need to call the API from within the notebook.

In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok

# Optional: set your ngrok auth token to avoid connection limits
# ngrok.set_auth_token("your_ngrok_authtoken")

tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url
print(f"✅ Public URL: {PUBLIC_URL}")
print(f"   API docs:   {PUBLIC_URL}/docs")

## Step 7 — Submit a checkpoint import job

`facebook/opt-125m` is a small public model — no HuggingFace token required.  
For gated models (Llama 3, Mistral, etc.), add `"hf_token": "hf_..."` to the request.

In [ ]:
import requests
import json

BASE = "http://localhost:8000"

resp = requests.post(f"{BASE}/v1/checkpoints/import", json={
    "source_path": "facebook/opt-125m",
    "target_name": "opt-125m-megatron",
    # "hf_token": "hf_..."  # uncomment for gated models
})

print("Response:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

job_id = resp.json()["job_id"]
print(f"\nJob ID: {job_id}")

## Step 8 — Stream logs in real-time

In [ ]:
import websocket
import threading
import json

def stream_logs(job_id):
    ws = websocket.WebSocket()
    ws.connect(f"ws://localhost:8000/v1/ws/jobs/{job_id}/logs")
    print(f"--- Log stream for job {job_id} ---")
    while True:
        msg = ws.recv()
        if not msg:
            break
        try:
            data = json.loads(msg)
            if data.get("type") == "stream_end":
                print(f"\n--- Stream ended (status: {data['status']}) ---")
                break
        except json.JSONDecodeError:
            print(msg)  # plain log line from the worker
    ws.close()

log_thread = threading.Thread(target=stream_logs, args=(job_id,), daemon=True)
log_thread.start()
print("Log stream started in background thread.")

## Step 9 — Poll job status and GPU telemetry

In [ ]:
import time

while True:
    resp = requests.get(f"{BASE}/v1/jobs/{job_id}")
    job  = resp.json()

    status   = job["status"]
    progress = job.get("progress") or {}

    print(f"Status: {status}", end="")

    if progress:
        step  = progress.get("step", "?")
        total = progress.get("total_steps", "?")
        loss  = progress.get("loss", "?")
        gpus  = progress.get("gpus", [])
        print(f" | Step: {step}/{total} | Loss: {loss}", end="")
        if gpus:
            g = gpus[0]
            print(
                f" | GPU: {g.get('util_pct')}% util "
                f"| {g.get('mem_used_gb'):.1f}/{g.get('mem_total_gb'):.0f} GB "
                f"| {g.get('temp_c')}°C",
                end=""
            )

    print()

    if status in ("completed", "failed", "cancelled"):
        if status == "failed":
            print(f"\n❌ Error: {job.get('error')}")
        else:
            print(f"\n✅ Job {status} successfully")
        break

    time.sleep(5)

## Step 10 — View registered checkpoints

In [ ]:
resp = requests.get(f"{BASE}/v1/checkpoints")
print(json.dumps(resp.json(), indent=2))

## Step 11 — Launch a LoRA fine-tuning job (optional)

Requires a completed Megatron-format checkpoint from Step 7 and a dataset directory.

In [ ]:
# Only run this after the import job (Step 7) completed successfully
# and you have a dataset at /content/datasets/my_data

resp = requests.post(f"{BASE}/v1/peft/lora", json={
    "pretrained_checkpoint": "/content/checkpoints/opt-125m-megatron",
    "dataset_path": "/content/datasets/my_data",
    "output_dir": "/content/checkpoints/opt-125m-lora",
    "num_gpus": 1,          # Colab gives 1 GPU
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_target_modules": ["linear_qkv", "linear_proj"],
})

print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

## Cleanup

In [ ]:
server_proc.terminate()
server_proc.wait()
print("✅ API server stopped")

try:
    from pyngrok import ngrok
    ngrok.kill()
    print("✅ ngrok tunnel closed")
except Exception:
    pass

## Known Colab Compatibility Notes

| Issue | Cause | Fix applied in this notebook |
|---|---|---|
| `megatron-core[dev,mlm]` resolution conflict | Dev extras clash with Colab env | Install `megatron-bridge --no-deps`, then `megatron-core` separately |
| `libcublas.so.13: not found` | Driver reports CUDA 13.0 but toolkit is 12.8 | Use `transformer-engine[core_cu12]` not `core_cu13` |
| `transformer-engine==2.13.0` incompatible | megatron-bridge 0.3.1 requires `<2.13.0` | Pin `>=2.10.0,<2.13.0` |
| `transformers==5.0.0` incompatible | megatron-bridge 0.3.1 requires `<5.0.0` | Pin `transformers<5.0.0` |
| `libcublas.so.12: not found` in worker | `LD_LIBRARY_PATH` not inherited by subprocess | Set before starting uvicorn, pass `env=os.environ.copy()` to `Popen` |